In [ ]:
import sys
from pathlib import Path

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

# -----------------------------------
# 1. FaithSRNet import
# -----------------------------------
sys.path.append(
    "/kaggle/input/models/muhammadhassan2/faithsrnet-v01/other/default/1"
)

from faithsrnet_v01 import FaithSRNetV01

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# -----------------------------------
# 2. Helper functions
# -----------------------------------
def pil_to_tensor(img):
    arr = np.asarray(img.convert("RGB"), dtype=np.float32) / 255.0
    return torch.from_numpy(arr).permute(2, 0, 1).contiguous()

def tensor_to_numpy(x):
    return (
        x.detach()
        .clamp(0, 1)
        .permute(1, 2, 0)
        .cpu()
        .numpy()
    )

def psnr(x, y):
    mse = F.mse_loss(x, y)
    return 10.0 * torch.log10(1.0 / (mse + 1e-12))

def gaussian_window(window_size=11, sigma=1.5, channels=3, device="cpu"):
    coords = torch.arange(window_size, device=device, dtype=torch.float32)
    coords = coords - window_size // 2

    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g = g / g.sum()

    w2 = torch.outer(g, g)

    return w2.expand(
        channels, 1, window_size, window_size
    ).contiguous()

def ssim(x, y, window_size=11):
    c = x.shape[1]

    window = gaussian_window(
        window_size,
        1.5,
        c,
        x.device
    )

    pad = window_size // 2

    mu_x = F.conv2d(x, window, padding=pad, groups=c)
    mu_y = F.conv2d(y, window, padding=pad, groups=c)

    sigma_x = (
        F.conv2d(x * x, window, padding=pad, groups=c)
        - mu_x * mu_x
    )

    sigma_y = (
        F.conv2d(y * y, window, padding=pad, groups=c)
        - mu_y * mu_y
    )

    sigma_xy = (
        F.conv2d(x * y, window, padding=pad, groups=c)
        - mu_x * mu_y
    )

    c1 = 0.01 ** 2
    c2 = 0.03 ** 2

    score = (
        (2 * mu_x * mu_y + c1) *
        (2 * sigma_xy + c2)
    ) / (
        (mu_x ** 2 + mu_y ** 2 + c1) *
        (sigma_x + sigma_y + c2)
        + 1e-12
    )

    return score.mean()

# -----------------------------------
# 3. Load trained BASE model
# -----------------------------------

checkpoint_path = (
    "/kaggle/input/models/muhammadhassan2/best-base/other/default/1/best.pth"
)

ckpt = torch.load(
    checkpoint_path,
    map_location=device
)

model = FaithSRNetV01(
    scale=4,
    use_degradation_adaptation=False,
    use_observation_consistency=False
).to(device)

model.load_state_dict(ckpt["model"])

model.eval()

print("Loaded checkpoint from epoch:", ckpt["epoch"])
print(
    "Best complex PSNR:",
    ckpt["best_complex_psnr"]
)

# -----------------------------------
# 4. Select test image
# -----------------------------------

image_path = (
    "/kaggle/input/datasets/"
    "nguyenquangnhat2100/"
    "ucmerced-landuse-dataset/"
    "UCMerced_LandUse/"
    "airplane/"
    "airplane00.tif"
)

img = Image.open(image_path).convert("RGB")

# Use fixed 224x224 center crop
w, h = img.size

left = (w - 224) // 2
top = (h - 224) // 2

img = img.crop(
    (
        left,
        top,
        left + 224,
        top + 224
    )
)

hr = pil_to_tensor(img).unsqueeze(0).to(device)

# -----------------------------------
# 5. Create x4 LR
# -----------------------------------

lr = F.interpolate(
    hr,
    scale_factor=0.25,
    mode="bicubic",
    align_corners=False,
    antialias=True
)

# Bicubic baseline back to HR size
bicubic = F.interpolate(
    lr,
    size=hr.shape[-2:],
    mode="bicubic",
    align_corners=False,
    antialias=True
)

# -----------------------------------
# 6. FaithSR inference
# -----------------------------------

with torch.no_grad():
    sr = model(lr).clamp(0, 1)

# -----------------------------------
# 7. Metrics
# -----------------------------------

bicubic_psnr = psnr(bicubic, hr).item()
faith_psnr = psnr(sr, hr).item()

bicubic_ssim = ssim(
    bicubic.float(),
    hr.float()
).item()

faith_ssim = ssim(
    sr.float(),
    hr.float()
).item()

print("\n--- Single Image Result ---")

print(
    f"Bicubic PSNR : {bicubic_psnr:.3f} dB"
)
print(
    f"FaithSR PSNR : {faith_psnr:.3f} dB"
)

print(
    f"Bicubic SSIM : {bicubic_ssim:.4f}"
)
print(
    f"FaithSR SSIM : {faith_ssim:.4f}"
)

print(
    f"\nPSNR Gain: "
    f"{faith_psnr - bicubic_psnr:+.3f} dB"
)

# -----------------------------------
# 8. Display
# -----------------------------------

lr_display = F.interpolate(
    lr,
    size=hr.shape[-2:],
    mode="nearest"
)

plt.figure(figsize=(18, 5))

plt.subplot(1, 4, 1)
plt.imshow(
    tensor_to_numpy(lr_display[0])
)
plt.title("Low Resolution x4")
plt.axis("off")

plt.subplot(1, 4, 2)
plt.imshow(
    tensor_to_numpy(bicubic[0])
)
plt.title(
    f"Bicubic\n"
    f"PSNR {bicubic_psnr:.2f}"
)
plt.axis("off")

plt.subplot(1, 4, 3)
plt.imshow(
    tensor_to_numpy(sr[0])
)
plt.title(
    f"FaithSR BASE\n"
    f"PSNR {faith_psnr:.2f}"
)
plt.axis("off")

plt.subplot(1, 4, 4)
plt.imshow(
    tensor_to_numpy(hr[0])
)
plt.title("Original HR")
plt.axis("off")

plt.tight_layout()
plt.show()